# Smart Elevator CV — YOLOv8 Training and Class-Based Area Estimation

End-to-end Colab notebook that:

1. Backs up any existing `best.pt` / `best_v2.pt` on Google Drive (versioned, dated filename).
2. Fine-tunes a fresh YOLOv8-s detector on the unified **three-class** dataset (`stroller`, `luggage`, `box`). `person` is no longer detected here — the dedicated head model `best_head.pt` supplies the person count at runtime.
3. Evaluates it on the held-out test split.
4. Runs a **class-based area estimator** on the test set and reports per-image and aggregate occupancy statistics.

## Why drop `person` from this model?

The four-class v2 detector duplicated the work already done by `best_head.pt` (head detection). To remove that duplication and the associated double-counting risk in `scripts/run_simulation.py`, the object detector is reduced to the three non-person cabin objects. At inference time, person counts come exclusively from the head model.

## Why class-count × average footprint?

Elevator CCTV is mounted in a top corner with a fish-eye lens; only heads and upper bodies of nearby occupants are visible. Full-body bounding boxes are therefore **physically impossible**, so we cannot derive floor area from box pixels.

Instead, every detection contributes a fixed standard footprint $\bar{a}_c$ taken from industry-standard cabin capacity values:

$$
A_\mathrm{occupied} = \sum_{c \in \mathcal{C}} n_c \cdot \bar{a}_c,
\qquad
\rho = \min\!\left(\frac{A_\mathrm{occupied}}{A_\mathrm{cabin}},\ 1\right)
$$

with $\bar{a}_\mathrm{person} = 0.20\,\mathrm{m^2}$ as the conventional area allowance per cabin passenger (applied downstream from the head detector); the remaining values are taken from typical product footprints. A position-aware variant (homography-based union of disks, or a birds-eye-view occupancy mask) is identified as future work in §5.5 of the thesis.

## Prerequisites (run once on your local machine)

1. Re-unify the dataset locally so it has only the three target classes:
   `python -m scripts.prepare_dataset --raw <RAW_DIR> --out data/unified`
2. (Optional) re-augment: `python scripts/augment_dataset.py`
3. `python -m scripts.package_for_colab` → produces fresh `code.zip` and `dataset.zip` under `Desktop/colab_upload/`.
4. Upload both ZIPs to `Google Drive / MyDrive / Capstone /` (overwrite the old ones).
5. In Drive, right-click this notebook → *Open with* → *Google Colaboratory*.

## 1. Mount Drive and extract the project + dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, zipfile, shutil
from datetime import date

BASE_DRIVE = '/content/drive/MyDrive/Capstone'
WORK = '/content/work'
REPO = f'{WORK}/Capstone-Project'
DATA = f'{WORK}/data/unified'
TODAY = date.today().isoformat()

assert os.path.exists(f'{BASE_DRIVE}/code.zip'),    f'code.zip not found at: {BASE_DRIVE}/code.zip'
assert os.path.exists(f'{BASE_DRIVE}/dataset.zip'), f'dataset.zip not found at: {BASE_DRIVE}/dataset.zip'

os.makedirs(REPO, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

with zipfile.ZipFile(f'{BASE_DRIVE}/code.zip') as z:
    z.extractall(REPO)
with zipfile.ZipFile(f'{BASE_DRIVE}/dataset.zip') as z:
    z.extractall(DATA)

# Rewrite the dataset's `path` field with the Colab-side absolute path.
# The local data.yaml carries a Windows path that Colab cannot resolve.
import yaml
yaml_path = f'{DATA}/data.yaml'
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)
old_path = data_cfg.get('path')
data_cfg['path'] = DATA
with open(yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(f'data.yaml path: {old_path!r} -> {DATA!r}')

%cd {REPO}
!ls

## 2. Back up the previous models (`best.pt`, `best_v2.pt`) on Drive

Copies any existing v1 / v2 weights to `MyDrive/Capstone/models/backups/`. Idempotent: re-running won't overwrite a previous backup.

In [ ]:
from pathlib import Path

# (tag, path-on-Drive) — anything that exists gets copied to backups/<tag>_backup_<DATE>.pt
backup_candidates = [
    ('best_v1_runs',    f'{BASE_DRIVE}/models/runs/elevator_v1/weights/best.pt'),
    ('best_v1_weights', f'{BASE_DRIVE}/models/weights/best.pt'),
    ('best_v2_runs',    f'{BASE_DRIVE}/models/runs/elevator_v2/weights/best.pt'),
    ('best_v2_weights', f'{BASE_DRIVE}/models/weights/best_v2.pt'),
]
found = [(tag, p) for tag, p in backup_candidates if os.path.exists(p)]

print('Discovered weights to back up:')
for tag, p in found:
    sz_mb = os.path.getsize(p) / 1e6
    print(f'  - [{tag}] {p}  ({sz_mb:.1f} MB)')

if not found:
    print('\nNo existing weights found — nothing to back up. Continuing to training.')
else:
    backup_dir = f'{BASE_DRIVE}/models/backups'
    os.makedirs(backup_dir, exist_ok=True)
    for tag, src in found:
        dst = f'{backup_dir}/{tag}_backup_{TODAY}.pt'
        if os.path.exists(dst):
            print(f'  [skip] backup already exists: {dst}')
        else:
            shutil.copy2(src, dst)
            print(f'  [ok]   backed up to: {dst}')

print('\nBackup directory contents:')
!ls -lh {BASE_DRIVE}/models/backups/ 2>/dev/null || echo '(directory does not exist yet)'

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. GPU and dataset audit

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi 2>/dev/null | head -15

from src.dataset.audit import audit_yolo_dataset, print_audit
from src.dataset.unify import TARGET_CLASSES
print_audit(audit_yolo_dataset(DATA), class_names=TARGET_CLASSES)

## 5. Train the model (run name: `elevator_v3`)

Augmentation has already been applied locally and idempotently. YOLOv8's runtime augmentation remains enabled with default settings.

The change from v2 → v3 is purely in the **label space**: `person` is no longer a target class, so v3 is a strict-subset detector compared to v2. Hyperparameters are kept identical so the only changed variable is the class scheme.

In [ ]:
from ultralytics import YOLO

VARIANT = 'yolov8s.pt'
EPOCHS = 100
BATCH = 32
IMGSZ = 640

RUNS_DIR = f'{BASE_DRIVE}/models/runs'
os.makedirs(RUNS_DIR, exist_ok=True)

model = YOLO(VARIANT)
results = model.train(
    data=f'{DATA}/data.yaml',
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    lr0=0.01,
    patience=25,
    seed=42,
    project=RUNS_DIR,
    name='elevator_v3',
    plots=True,
    save_period=10,
    device=0 if torch.cuda.is_available() else 'cpu',
)
BEST_V3 = f'{results.save_dir}/weights/best.pt'
print('best.pt (v3):', BEST_V3)

## 6. Evaluate on the held-out test split

In [ ]:
from ultralytics import YOLO
model_v3 = YOLO(BEST_V3)
metrics = model_v3.val(data=f'{DATA}/data.yaml', split='test')
print('mAP50:    ', metrics.box.map50)
print('mAP50-95: ', metrics.box.map)
print('Per-class mAP50:', dict(zip(metrics.names.values(), metrics.box.maps.tolist())))

## 7. Save `best.pt` to Drive and the working repo

Writes the new model under a versioned filename (`best_v3.pt`) while keeping the v1/v2 backups intact. The unversioned `best.pt` always points to the latest model so downstream demos pick it up automatically.

In [ ]:
drive_v3 = f'{BASE_DRIVE}/models/weights/best_v3.pt'
drive_active = f'{BASE_DRIVE}/models/weights/best.pt'
repo_dst = f'{REPO}/models/weights/best.pt'

for d in (drive_v3, drive_active, repo_dst):
    os.makedirs(os.path.dirname(d), exist_ok=True)
    shutil.copy2(BEST_V3, d)
    print('copied:', d, f'({os.path.getsize(d) / 1e6:.1f} MB)')

## 8. Class-based area estimator

### 8.1 Method

For each detection $d$ with class $c_d$, we add a fixed standard footprint $\bar{a}_{c_d}$ to a running total. Total occupied area and occupancy ratio:

$$
A_\mathrm{occupied} = \sum_{i} \bar{a}_{c_i},
\qquad
\rho = \min\!\left(\frac{A_\mathrm{occupied}}{A_\mathrm{cabin}},\ 1\right)
$$

### 8.2 Per-class average footprint values (this detector only)

| Class | $\bar{a}_c$ (m²) | Notes |
|---|---|---|
| **stroller** | **0.45** | Mid-range single pushchair (≈ 90 × 50 cm) including the occupant child. |
| **luggage** | **0.20** | Mid-size cabin / check-in suitcase footprint (≈ 56 × 36 cm). |
| **box** | **0.20** | Average e-commerce / logistics carton footprint (≈ 50 × 40 cm). |

> `person` (0.20 m²) is **not** in this table — `scripts/run_simulation.py` and `demo/app.py` add the person contribution from the head model after fusing the two detectors' outputs.

> **Limitation.** This estimator does not use detection positions and therefore cannot tell whether two objects overlap in image space. When $A_\mathrm{occupied} > A_\mathrm{cabin}$ we clamp to $\rho = 1$. A position-aware extension (homography projection + per-class footprint union, or a birds-eye-view occupancy mask) is identified as future work in §5.5 of the thesis.

In [ ]:
from dataclasses import dataclass

# Average footprints (m^2). Three-class detector only — `person` is omitted
# because this model no longer detects it (handled by best_head.pt downstream).
CLASS_AREAS_M2 = {
    'stroller': 0.45,
    'luggage':  0.20,
    'box':      0.20,
}

# Cabin floor area — defaults match configs/default.yaml (1.4 x 1.6 m).
ELEVATOR_FLOOR_AREA_M2 = 2.24

# Bypass threshold — defaults to thresholds.area_bypass_ratio in configs.
AREA_BYPASS_RATIO = 0.90


@dataclass
class AreaReport:
    counts: dict           # {'stroller': 1, ...}
    occupied_m2: float     # total estimated occupied area
    cabin_m2: float        # cabin floor area
    occupancy_ratio: float # in [0, 1]
    breakdown_m2: dict     # {'stroller': 0.45, ...}
    bypass: bool           # area bypass threshold reached?

    def pretty(self) -> str:
        lines = [
            f'Occupancy: {self.occupied_m2:.2f} / {self.cabin_m2:.2f} m^2 '
            f'({self.occupancy_ratio*100:.1f}%)'
        ]
        for cls, n in self.counts.items():
            lines.append(f'  - {cls:<9} x{n}  ->  {self.breakdown_m2[cls]:.2f} m^2')
        if self.bypass:
            lines.append(f'  [BYPASS] occupancy >= {AREA_BYPASS_RATIO*100:.0f}% — reject hall call')
        return '\n'.join(lines)


def estimate_area(
    detections,
    class_areas_m2: dict = CLASS_AREAS_M2,
    cabin_m2: float = ELEVATOR_FLOOR_AREA_M2,
) -> AreaReport:
    """Class-count x average-footprint occupancy estimate (no person)."""
    counts: dict[str, int] = {}
    breakdown: dict[str, float] = {}
    occupied = 0.0

    for det in detections:
        cls = det.class_name
        avg = class_areas_m2.get(cls, 0.0)
        if avg <= 0:
            continue
        counts[cls] = counts.get(cls, 0) + 1
        breakdown[cls] = breakdown.get(cls, 0.0) + avg
        occupied += avg

    ratio = min(occupied / cabin_m2, 1.0) if cabin_m2 > 0 else 0.0
    return AreaReport(
        counts=counts,
        occupied_m2=occupied,
        cabin_m2=cabin_m2,
        occupancy_ratio=ratio,
        breakdown_m2=breakdown,
        bypass=(ratio >= AREA_BYPASS_RATIO),
    )


# Sanity check with synthetic detections.
from collections import namedtuple
FakeDet = namedtuple('FakeDet', 'class_name')
fake = [FakeDet('stroller')] * 1 + [FakeDet('luggage')] * 2 + [FakeDet('box')] * 1
print(estimate_area(fake).pretty())

## 9. Demo inference + area report (six test images)

For each image we display the YOLO detections and overlay the estimated occupancy ratio in the title.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import glob
from src.detection.detector import Detection

test_imgs = sorted(glob.glob(f'{DATA}/test/images/*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, test_imgs):
    res = model_v3.predict(img_path, conf=0.4, verbose=False)[0]

    # ultralytics result -> Detection list
    dets = []
    if res.boxes is not None:
        for box in res.boxes:
            cls_id = int(box.cls.item())
            cls_name = res.names[cls_id]
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            dets.append(Detection(
                class_id=cls_id,
                class_name=cls_name,
                confidence=float(box.conf.item()),
                bbox=(int(x1), int(y1), int(x2), int(y2)),
            ))

    report = estimate_area(dets)

    ax.imshow(Image.fromarray(res.plot()[..., ::-1]))
    title = (
        os.path.basename(img_path)[:32] + '\n' +
        f'occupancy: {report.occupancy_ratio*100:.1f}% '
        f'({report.occupied_m2:.2f}/{report.cabin_m2:.2f} m²)'
        + ('  [BYPASS]' if report.bypass else '')
    )
    ax.set_title(title, fontsize=9)
    ax.axis('off')

    print(f'\n=== {os.path.basename(img_path)} ===')
    print(report.pretty())

plt.tight_layout()
plt.show()

## 10. Aggregate area report over the full test set

Number of frames that trigger bypass, mean / median / max occupancy, and the per-class detection distribution. The histogram visualizes occupancy frequencies.

In [ ]:
import numpy as np

all_test = sorted(glob.glob(f'{DATA}/test/images/*.jpg'))
ratios = []
bypass_count = 0
class_total = {k: 0 for k in CLASS_AREAS_M2}

for img_path in all_test:
    res = model_v3.predict(img_path, conf=0.4, verbose=False)[0]
    dets = []
    if res.boxes is not None:
        for box in res.boxes:
            cls_id = int(box.cls.item())
            cls_name = res.names[cls_id]
            dets.append(Detection(
                class_id=cls_id, class_name=cls_name,
                confidence=float(box.conf.item()),
                bbox=(0, 0, 1, 1),
            ))
    rep = estimate_area(dets)
    ratios.append(rep.occupancy_ratio)
    if rep.bypass:
        bypass_count += 1
    for k, v in rep.counts.items():
        class_total[k] = class_total.get(k, 0) + v

ratios = np.array(ratios)
print(f'Test images:           {len(ratios)}')
print(f'Mean occupancy:        {ratios.mean()*100:.1f}%')
print(f'Median occupancy:      {np.median(ratios)*100:.1f}%')
print(f'Max occupancy:         {ratios.max()*100:.1f}%')
print(f'Frames triggering bypass: {bypass_count} / {len(ratios)} '
      f'({bypass_count/len(ratios)*100:.1f}%)')
print(f'\nTotal detections by class:')
for k, v in class_total.items():
    print(f'  - {k:<9} x{v}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ratios * 100, bins=20, edgecolor='black', alpha=0.75)
ax.axvline(AREA_BYPASS_RATIO * 100, color='red', linestyle='--',
           label=f'bypass threshold ({AREA_BYPASS_RATIO*100:.0f}%)')
ax.set_xlabel('Occupancy (%)')
ax.set_ylabel('Frame count')
ax.set_title('Estimated cabin occupancy distribution on the test set (no person)')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Future work

1. **Homography calibration.** Mark the four cabin-floor corners and compute the homography with `cv2.findHomography` to drive a position-aware occupancy estimator (union of per-class disks on the floor plane, or a birds-eye-view occupancy mask) instead of constant per-class footprints.
2. **Pose estimation.** With an additional ceiling camera, YOLOv8-pose can extract full-body keypoints; the silhouette mask replaces the bbox approximation.
3. **Multi-frame tracking.** A track-ID assigned by BoT-SORT or ByteTrack prevents the same object from being counted twice across consecutive frames.
4. **Per-class footprint distribution.** Instead of a fixed mean, model class-internal variance (large vs. compact stroller, cabin vs. check-in luggage) as separate sub-classes.
5. **Label policy.** When collecting future data, adopt a consistent labeling policy so bbox-area-based methods become camera-independent.

## 12. Use the trained model locally

1. The new `best_v3.pt` is automatically synced to your local machine via Drive: `Desktop/Capstone-Project/models/weights/best.pt` (it overwrites the active pointer, but `best_v2.pt` and `best_v3.pt` are also kept on Drive under `models/weights/` and in `models/backups/`).
2. Notebook 04 (energy simulation) and `scripts/run_simulation.py` consume this `best.pt` directly. They run in **hybrid mode** with `best_head.pt` supplying the person count — that hybrid logic is what motivated dropping `person` from this detector.
3. To roll back, copy any file from `MyDrive/Capstone/models/backups/best_v2_*.pt` (or `best_v1_*.pt`) over the active `best.pt`.